# Bandpower Feature Augmentation for MEG Classification

This notebook extends `load_and_prep_data.ipynb` by computing **per-sensor band-power features** from each sliding window.

## Why bandpower?
The four brain states differ most clearly in their **spectral profiles**:
- `rest` → dominant alpha (8–13 Hz), suppressed beta
- `task_motor` → beta desynchronisation (13–30 Hz) + gamma bursts
- `task_story_math` → theta elevation (4–8 Hz) in frontal sensors
- `task_working_memory` → theta + gamma increase, alpha suppression

Using Welch's method on each 2-second window we extract 5 band-power values per sensor,
giving a `(248, 5)` feature matrix per sample.

This provides two benefits:
1. **Data augmentation**: every raw window produces *both* a raw-timeseries sample and a
   bandpower-feature sample — effectively doubling training set size.
2. **Richer signal**: bandpower features are compact, less noisy, and well-suited to EEGNet's
   depthwise spatial convolutions.

---
### Frequency bands used (following the article)
| Band | Range (Hz) |
|------|------------|
| Delta | 0.5 – 4 |
| Theta | 4 – 8 |
| Alpha | 8 – 13 |
| Beta | 13 – 30 |
| Gamma | 30 – 60 |

### PSD formula (Welch's method)
$$PSD(f) = \frac{|\mathcal{F}\{x(t)\}|^2}{T}$$
Band power is the integral of PSD over the band:
$$P_{band} = \int_{f_{low}}^{f_{high}} PSD(f) \, df$$
approximated via the **trapezoidal rule** over Welch frequency bins.

## 1. Imports & Constants

In [ ]:
import os
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import seaborn as sns

# ── Directory produced by load_and_prep_data.ipynb ──
DATA_DIR   = 'Preprocessed data'
OUTPUT_DIR = 'Bandpower data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Signal parameters (must match load_and_prep_data.ipynb) ──
FS = 254.0          # approximate sampling frequency after ÷8 downsampling

# ── Frequency bands (Hz) following the BITS Pilani Neurotech article ──
BANDS = {
    'delta': (0.5, 4),
    'theta': (4,   8),
    'alpha': (8,  13),
    'beta':  (13, 30),
    'gamma': (30, 60),
}
BAND_NAMES = list(BANDS.keys())
N_BANDS    = len(BANDS)

CLASSES = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']
print(f'Sampling rate: {FS} Hz | Bands: {BAND_NAMES}')

## 2. Load Pre-processed Windowed Data

These `.npy` files were produced by the sliding-window step in `load_and_prep_data.ipynb`.
Each `X` array has shape `(N_windows, 248, 500)` — N windows of 2 s each.

In [ ]:
def load(name):
    path = os.path.join(DATA_DIR, f'{name}.npy')
    arr = np.load(path)
    print(f'  {name}: {arr.shape}')
    return arr

print('Loading windowed splits...')
X_intra_train_w  = load('X_intra_train_w')
y_intra_train_w  = load('y_intra_train_w')
X_intra_test_w   = load('X_intra_test_w')
y_intra_test_w   = load('y_intra_test_w')

X_cross_train_w  = load('X_cross_train_w')
y_cross_train_w  = load('y_cross_train_w')
X_cross_test1_w  = load('X_cross_test1_w')
y_cross_test1_w  = load('y_cross_test1_w')
X_cross_test2_w  = load('X_cross_test2_w')
y_cross_test2_w  = load('y_cross_test2_w')
X_cross_test3_w  = load('X_cross_test3_w')
y_cross_test3_w  = load('y_cross_test3_w')

## 3. Bandpower Extraction

For each window `(248, T)` we compute **Welch's PSD** per sensor, then integrate the PSD
within each frequency band using the trapezoidal rule.

Result shape per window: `(248, 5)` — 248 sensors × 5 bands.

We also compute **relative** bandpower (each band divided by total power) to
remove amplitude scale differences between subjects.

In [ ]:
def compute_bandpower_window(window, fs=FS, bands=BANDS, relative=False):
    """
    Compute per-sensor band power for one window.

    Parameters
    ----------
    window : np.ndarray  shape (n_sensors, n_times)
    fs     : float       sampling frequency in Hz
    bands  : dict        {name: (f_low, f_high)}
    relative : bool      if True, divide each band by total power

    Returns
    -------
    features : np.ndarray  shape (n_sensors, n_bands)
    """
    n_sensors, n_times = window.shape
    # nperseg: use half the window or 256, whichever is smaller
    nperseg = min(256, n_times // 2)

    features = np.zeros((n_sensors, len(bands)), dtype=np.float32)

    for s in range(n_sensors):
        # Welch's PSD estimate — shape: (n_freqs,)
        freqs, psd = signal.welch(window[s], fs=fs, nperseg=nperseg)

        total_power = np.trapz(psd, freqs) + 1e-12  # avoid /0

        for b_idx, (band_name, (f_lo, f_hi)) in enumerate(bands.items()):
            mask = (freqs >= f_lo) & (freqs <= f_hi)
            # trapezoidal integration over the band
            band_power = np.trapz(psd[mask], freqs[mask]) if mask.sum() > 1 else 0.0
            if relative:
                band_power /= total_power
            features[s, b_idx] = band_power

    return features


def extract_bandpower(X, fs=FS, bands=BANDS, relative=False, verbose=True):
    """
    Extract band-power features for all windows in X.

    Parameters
    ----------
    X : np.ndarray  shape (N, n_sensors, n_times)

    Returns
    -------
    X_bp : np.ndarray  shape (N, n_sensors, n_bands)
    """
    N = X.shape[0]
    n_sensors = X.shape[1]
    n_bands = len(bands)
    X_bp = np.zeros((N, n_sensors, n_bands), dtype=np.float32)

    for i in range(N):
        X_bp[i] = compute_bandpower_window(X[i], fs=fs, bands=bands, relative=relative)
        if verbose and (i + 1) % max(1, N // 10) == 0:
            print(f'  processed {i+1}/{N} windows', end='\r')

    if verbose:
        print(f'  done — shape: {X_bp.shape}')
    return X_bp


# Quick sanity check on one window
sample_bp = compute_bandpower_window(X_intra_train_w[0])
print('Single window bandpower shape:', sample_bp.shape)   # should be (248, 5)
print('Band means across sensors:')
for i, name in enumerate(BAND_NAMES):
    print(f'  {name:6s}: {sample_bp[:, i].mean():.4f}')

## 4. Extract & Save Bandpower Features

We compute **both absolute and relative** bandpower, giving two feature sets:
- `X_*_bp_abs` — raw power (µT²/Hz integrated)
- `X_*_bp_rel` — fraction of total power per sensor

Relative bandpower is more robust to inter-subject amplitude differences and is
recommended for cross-subject classification.

In [ ]:
splits = [
    ('intra_train', X_intra_train_w),
    ('intra_test',  X_intra_test_w),
    ('cross_train', X_cross_train_w),
    ('cross_test1', X_cross_test1_w),
    ('cross_test2', X_cross_test2_w),
    ('cross_test3', X_cross_test3_w),
]

for split_name, X_split in splits:
    print(f'\n--- {split_name} ({X_split.shape[0]} windows) ---')

    # Absolute bandpower
    X_abs = extract_bandpower(X_split, relative=False)
    np.save(os.path.join(OUTPUT_DIR, f'X_{split_name}_bp_abs'), X_abs)

    # Relative bandpower
    X_rel = extract_bandpower(X_split, relative=True, verbose=False)
    np.save(os.path.join(OUTPUT_DIR, f'X_{split_name}_bp_rel'), X_rel)

    print(f'  saved abs {X_abs.shape} + rel {X_rel.shape}')

# Copy labels (same windows, same labels)
for name, y in [
    ('y_intra_train_w', y_intra_train_w),
    ('y_intra_test_w',  y_intra_test_w),
    ('y_cross_train_w', y_cross_train_w),
    ('y_cross_test1_w', y_cross_test1_w),
    ('y_cross_test2_w', y_cross_test2_w),
    ('y_cross_test3_w', y_cross_test3_w),
]:
    np.save(os.path.join(OUTPUT_DIR, name), y)

print('\nAll bandpower features saved to:', OUTPUT_DIR)

## 5. Exploratory Analysis of Bandpower Features

Verify that the features discriminate between classes before plugging into EEGNet.

In [ ]:
# Load the just-saved relative bandpower for intra train
X_bp_rel = np.load(os.path.join(OUTPUT_DIR, 'X_intra_train_bp_rel.npy'))
y_labels = y_intra_train_w

# Average across sensors for a (N, 5) summary
X_sensor_avg = X_bp_rel.mean(axis=1)   # (N, 5)

# ── Band-power distribution per class ──────────────────────────────────────
fig, axes = plt.subplots(1, N_BANDS, figsize=(16, 4), sharey=False)
for b_idx, band_name in enumerate(BAND_NAMES):
    ax = axes[b_idx]
    for cls_idx, cls_name in enumerate(CLASSES):
        mask = y_labels == cls_idx
        vals = X_sensor_avg[mask, b_idx]
        ax.hist(vals, bins=20, alpha=0.6, label=cls_name)
    ax.set_title(f'{band_name.capitalize()} band')
    ax.set_xlabel('Relative power')
    if b_idx == 0:
        ax.set_ylabel('Count')
    ax.legend(fontsize=7)
fig.suptitle('Relative band-power distributions by class (sensor-averaged, intra train)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bandpower_class_distributions.png'), dpi=120)
plt.show()

In [ ]:
# ── Class-mean bandpower heatmap (sensor × band) ───────────────────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 5))
for cls_idx, cls_name in enumerate(CLASSES):
    mask = y_labels == cls_idx
    mean_map = X_bp_rel[mask].mean(axis=0)   # (248, 5)
    im = axes[cls_idx].imshow(mean_map, aspect='auto', cmap='viridis',
                               vmin=0, vmax=X_bp_rel.mean(axis=0).max())
    axes[cls_idx].set_title(cls_name.replace('task_', ''))
    axes[cls_idx].set_xlabel('Band')
    axes[cls_idx].set_xticks(range(N_BANDS))
    axes[cls_idx].set_xticklabels(BAND_NAMES, rotation=30)
    if cls_idx == 0:
        axes[cls_idx].set_ylabel('Sensor index')

fig.colorbar(im, ax=axes[-1], label='Relative power')
fig.suptitle('Mean relative band-power map per class (sensors × bands)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bandpower_heatmaps.png'), dpi=120)
plt.show()

## 6. EEGNet on Bandpower Features

The bandpower array has shape `(N, 248, 5)` — same spatial dimension (248 sensors) but
only 5 time-like dimensions (one per band).  
This is a perfect input for **EEGNet's depthwise spatial convolution** block, which treats
the second axis as channels (sensors) and the third as the temporal dimension.

We adjust two hyperparameters:
- `n_timesteps = 5` (number of bands)
- `kern_len = 3` (temporal kernel ≤ n_timesteps)

Optionally, we **concatenate** the raw window and bandpower datasets to maximally
exploit both representations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ── EEGNet (same as EEGNet_MEG_Classification.ipynb, parameterised) ─────────
class EEGNet(nn.Module):
    def __init__(self, n_classes=4, n_channels=248, n_timesteps=5,
                 F1=8, D=2, F2=16, kern_len=3, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_len),
                      padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 2)),
            nn.Dropout(dropout),
        )
        sep_kern = min(3, max(1, n_timesteps // 2))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, kernel_size=(1, sep_kern),
                      padding=(0, sep_kern//2), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AdaptiveAvgPool2d((1, 1)),  # handles any remaining temporal size
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(F2, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)      # (B, 1, sensors, bands_or_time)
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# Sanity check with bandpower input shape
n_t = N_BANDS  # 5 bands
m = EEGNet(n_channels=248, n_timesteps=n_t, kern_len=3, dropout=0.5).to(DEVICE)
dummy = torch.zeros(4, 248, n_t).to(DEVICE)
out = m(dummy)
print('Output shape:', out.shape)   # should be [4, 4]
print('Parameters:', sum(p.numel() for p in m.parameters()))

In [ ]:
# ── Training utilities ──────────────────────────────────────────────────────
def make_loader(X, y, batch_size=32, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, opt, criterion):
    model.train()
    total_loss, correct = 0., 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); opt.step()
        total_loss += loss.item() * len(yb)
        correct += (model(xb).argmax(1) == yb).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0., 0
    preds, labels = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * len(yb)
        p = logits.argmax(1)
        correct += (p == yb).sum().item()
        preds.extend(p.cpu().numpy())
        labels.extend(yb.cpu().numpy())
    n = len(loader.dataset)
    return total_loss / n, correct / n, np.array(preds), np.array(labels)

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3, patience=15):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state, no_improve = 0., None, 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, opt, criterion)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 20 == 0 or epoch == 1:
            print(f'Epoch {epoch:4d}/{epochs} | '
                  f'train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
                  f'val   loss={va_loss:.4f} acc={va_acc:.3f}')

        if no_improve >= patience:
            print(f'Early stop at epoch {epoch}  (best val acc={best_acc:.4f})')
            break

    model.load_state_dict(best_state)
    return history, best_acc

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'],   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    axes[1].axhline(0.25, ls='--', color='grey', label='chance')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_confusion(preds, labels, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(title)
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Intra-Subject — Bandpower Features

In [ ]:
X_bp_train = np.load(os.path.join(OUTPUT_DIR, 'X_intra_train_bp_rel.npy'))
y_bp_train = np.load(os.path.join(OUTPUT_DIR, 'y_intra_train_w.npy'))
X_bp_test  = np.load(os.path.join(OUTPUT_DIR, 'X_intra_test_bp_rel.npy'))
y_bp_test  = np.load(os.path.join(OUTPUT_DIR, 'y_intra_test_w.npy'))

print(f'Intra train: {X_bp_train.shape}  test: {X_bp_test.shape}')

bp_train_loader = make_loader(X_bp_train, y_bp_train, batch_size=32)
bp_test_loader  = make_loader(X_bp_test,  y_bp_test,  batch_size=32, shuffle=False)

bp_intra_model = EEGNet(
    n_classes=4, n_channels=248, n_timesteps=N_BANDS,
    kern_len=3, dropout=0.5
).to(DEVICE)

print('=== Intra-Subject (bandpower features) ===')
bp_intra_hist, bp_intra_best = train_model(
    bp_intra_model, bp_train_loader, bp_test_loader, epochs=150, lr=1e-3
)
print(f'Best intra-subject (bandpower) test accuracy: {bp_intra_best:.4f}')

In [ ]:
plot_history(bp_intra_hist, 'Intra-Subject — Bandpower Features')
criterion = nn.CrossEntropyLoss()
_, acc, preds, labels = evaluate(bp_intra_model, bp_test_loader, criterion)
print(f'Intra-subject (bandpower) test accuracy: {acc:.4f}\n')
print(classification_report(labels, preds, target_names=CLASSES))
plot_confusion(preds, labels, 'Intra-Subject (Bandpower) Confusion Matrix')

## 8. Cross-Subject — Relative Bandpower Features

Relative bandpower normalises away individual amplitude differences,
making it especially useful for cross-subject generalisation.

In [ ]:
X_cross_bp_train = np.load(os.path.join(OUTPUT_DIR, 'X_cross_train_bp_rel.npy'))
y_cross_bp_train = np.load(os.path.join(OUTPUT_DIR, 'y_cross_train_w.npy'))
X_cross_bp_t1 = np.load(os.path.join(OUTPUT_DIR, 'X_cross_test1_bp_rel.npy'))
y_cross_bp_t1 = np.load(os.path.join(OUTPUT_DIR, 'y_cross_test1_w.npy'))
X_cross_bp_t2 = np.load(os.path.join(OUTPUT_DIR, 'X_cross_test2_bp_rel.npy'))
y_cross_bp_t2 = np.load(os.path.join(OUTPUT_DIR, 'y_cross_test2_w.npy'))
X_cross_bp_t3 = np.load(os.path.join(OUTPUT_DIR, 'X_cross_test3_bp_rel.npy'))
y_cross_bp_t3 = np.load(os.path.join(OUTPUT_DIR, 'y_cross_test3_w.npy'))

cross_bp_train_loader = make_loader(X_cross_bp_train, y_cross_bp_train, batch_size=32)
cross_bp_t1_loader    = make_loader(X_cross_bp_t1, y_cross_bp_t1, shuffle=False)
cross_bp_t2_loader    = make_loader(X_cross_bp_t2, y_cross_bp_t2, shuffle=False)
cross_bp_t3_loader    = make_loader(X_cross_bp_t3, y_cross_bp_t3, shuffle=False)

bp_cross_model = EEGNet(
    n_classes=4, n_channels=248, n_timesteps=N_BANDS,
    kern_len=3, dropout=0.5
).to(DEVICE)

print('=== Cross-Subject (bandpower features) ===')
bp_cross_hist, bp_cross_best = train_model(
    bp_cross_model, cross_bp_train_loader, cross_bp_t1_loader, epochs=150, lr=1e-3
)

criterion = nn.CrossEntropyLoss()
for i, (loader, y_true) in enumerate([
    (cross_bp_t1_loader, y_cross_bp_t1),
    (cross_bp_t2_loader, y_cross_bp_t2),
    (cross_bp_t3_loader, y_cross_bp_t3),
], start=1):
    _, acc, preds, labels = evaluate(bp_cross_model, loader, criterion)
    print(f'Cross-subject test{i} accuracy: {acc:.4f}')
    print(classification_report(labels, preds, target_names=CLASSES, zero_division=0))
    plot_confusion(preds, labels, f'Cross-Subject (Bandpower) Test {i}')

## 9. Combined Model — Raw + Bandpower

A simple **late-fusion** ensemble: train EEGNet on raw windows AND on bandpower
features separately, then average their softmax outputs at test time.
This is the easiest way to exploit both representations without changing either
model architecture.

In [ ]:
# Load the raw-window intra models from EEGNet_MEG_Classification.ipynb
# (run that notebook first; here we just show the fusion logic)

@torch.no_grad()
def ensemble_predict(raw_model, bp_model, raw_loader, bp_loader, raw_weight=0.5):
    """Average softmax probabilities from raw and bandpower models."""
    raw_model.eval(); bp_model.eval()
    all_preds, all_labels = [], []
    for (xr, yr), (xb, _) in zip(raw_loader, bp_loader):
        xr, xb = xr.to(DEVICE), xb.to(DEVICE)
        p_raw = F.softmax(raw_model(xr), dim=1)
        p_bp  = F.softmax(bp_model(xb),  dim=1)
        p_avg = raw_weight * p_raw + (1 - raw_weight) * p_bp
        preds = p_avg.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yr.numpy())
    preds_arr  = np.array(all_preds)
    labels_arr = np.array(all_labels)
    acc = (preds_arr == labels_arr).mean()
    return acc, preds_arr, labels_arr

# Example usage (uncomment once the raw intra_model is available):
# acc, preds, labels = ensemble_predict(
#     intra_model,      # from EEGNet_MEG_Classification.ipynb
#     bp_intra_model,   # trained above
#     intra_test_loader_raw,
#     bp_test_loader,
#     raw_weight=0.4,
# )
# print(f'Ensemble intra accuracy: {acc:.4f}')
# plot_confusion(preds, labels, 'Ensemble (Raw + Bandpower)')

print('Ensemble fusion cell ready — uncomment lines above once raw model is trained.')